In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════╗
║   NIFTY GAMMA SCALPING ENGINE  v4.0  —  REBUILT FROM FIRST PRINCIPLES║
╠══════════════════════════════════════════════════════════════════════╣
"""

import os, glob, re, warnings
from datetime import date, timedelta
from typing import Optional, List, Dict
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter
from scipy.stats import norm

warnings.filterwarnings('ignore')
pd.options.mode.chained_assignment = None

# ─────────────────────────────────────────────────────────────────────
#  PATHS  (edit these)
# ─────────────────────────────────────────────────────────────────────
DATA_1MIN  = '/content/drive/MyDrive/1MIN'   # folder with 7 CSV files
DATA_3MIN  = '/content/drive/MyDrive/3MIN'   # folder with 3 CSV files
OUT_DIR    = '/content/drive/MyDrive/gamma_v4_output'

# ─────────────────────────────────────────────────────────────────────
#  STRATEGY PARAMETERS
# ─────────────────────────────────────────────────────────────────────
P = {
  "IVR_MAX": 49.78531780525893,
  "IV_RV_MAX": 1.0372435519615906,
  "RV_WINDOW": 59,
  "IVR_DAYS": 23,
  "ER_WINDOW": 38,
  "ENTRY_START": "09:30",
  "ENTRY_END": "12:00",
  "SESSION_END": "15:10",
  "DTE_MIN": 3,
  "DTE_MAX": 4,
  "MAX_TRADES_PER_DAY": 5,
  "COOLDOWN_LOSS_MIN": 64,
  "STRADDLE_TP": 0.15537784628997814,
  "STRADDLE_SL": 0.13800004668430635,
  "IV_EXPAND_EXIT": 0.10081376275111595,
  "THETA_MAX_MIN": 88,
  "HEDGE_STEP_PTS": 51,
  "HEDGE_REVERT_PTS": 41,
  "HEDGE_SL_PTS": 171,
  "STRADDLE_COST": 120,
  "HEDGE_COST": 60,
  "LOT_SIZE": 65,
  "LOTS": 1,
  "RISK_FREE": 0.065,
  "MINS_PER_DAY": 375
}
#  {
#     # ── Entry ──────────────────────────────────────────────────────
#     'IVR_MAX'          : 50.0,    # IV Rank < 50%  (was 35.76 → too tight)
#     'IV_RV_MAX'        : 0.85,    # IV/RV < 0.85   (was 0.606 → only 18 trades)
#     'RV_WINDOW'        : 22,      # realized vol lookback (Optuna: 46% importance)
#     'IVR_DAYS'         : 13,      # IVR lookback days (Optuna best)
#     'ER_WINDOW'        : 26,      # Kaufman ER window (Optuna best)

#     # ── Time ───────────────────────────────────────────────────────
#     'ENTRY_START'      : '09:30', # include 09:30 (valid signals exist here)
#     'ENTRY_END'        : '13:30', # stop entries at 13:30
#     'SESSION_END'      : '15:10', # force close before 15:15

#     # ── DTE ────────────────────────────────────────────────────────
#     'DTE_MIN'          : 2,       # skip near-expiry (Optuna)
#     'DTE_MAX'          : 6,       # skip far expiry

#     # ── Trade limits ───────────────────────────────────────────────
#     'MAX_TRADES_PER_DAY': 3,      # max straddles per day
#     'COOLDOWN_LOSS_MIN' : 45,     # minutes to wait after a loss

#     # ── Straddle exit ──────────────────────────────────────────────
#     'STRADDLE_TP'      : 0.20,    # exit straddle at +20% gain  (gamma_win)
#     'STRADDLE_SL'      : 0.15,    # exit straddle at -15% loss  (max_drawdown)
#     'IV_EXPAND_EXIT'   : 0.05,    # exit if IV expands 5%       (vega_win)
#     # NO iv_crush exit — IV reverts, hold through it
#     'THETA_MAX_MIN'    : 180,     # max hold time (3 hours)

#     # ── Hedge parameters ───────────────────────────────────────────
#     # Hedge fires when spot moves HEDGE_STEP pts from last hedge level
#     # This is pure delta-hedge math — no option lookup required
#     'HEDGE_STEP_PTS'   : 50,      # hedge trigger: every 50 Nifty points
#     'HEDGE_REVERT_PTS' : 25,      # close hedge when spot reverts within 25 pts
#     'HEDGE_SL_PTS'     : 100,     # close hedge if spot moves 100pts against it

#     # ── Costs ──────────────────────────────────────────────────────
#     'STRADDLE_COST'    : 120,     # Rs120 all-in per straddle (entry+exit)
#     'HEDGE_COST'       : 60,      # Rs60 per hedge round-trip (entry+exit)

#     # ── Position ───────────────────────────────────────────────────
#     'LOT_SIZE'         : 65,      # current Nifty lot size
#     'LOTS'             : 1,

#     # ── Constants ──────────────────────────────────────────────────
#     'RISK_FREE'        : 0.065,
#     'MINS_PER_DAY'     : 375,
# }


# ═════════════════════════════════════════════════════════════════════
#  DATA LOADING
# ═════════════════════════════════════════════════════════════════════

def _clean_date(s):
    return str(s).replace('="','').replace('"','').replace('=','').strip()

def _offset_num(s):
    s = str(s).strip().upper()
    if s == 'ATM': return 0
    m = re.search(r'ATM([+-]\d+)', s)
    return int(m.group(1)) if m else None

def load_and_prepare(folder: str, label: str, keep: int = 2) -> pd.DataFrame:
    """Load all CSVs, parse, keep only ATM ± keep strikes."""
    files = sorted(glob.glob(os.path.join(folder, '*.csv')))
    if not files:
        raise FileNotFoundError(f"No CSVs in {folder}")

    print(f"\n{'─'*58}")
    print(f"  {label}: {len(files)} files from {folder}")
    frames = []
    for f in files:
        df = pd.read_csv(f, dtype={'date': str, 'time': str}, low_memory=False)
        df.columns = [c.strip().lower() for c in df.columns]
        frames.append(df)
        print(f"  ✓ {os.path.basename(f):30s} {len(df):>10,} rows")

    raw = pd.concat(frames, ignore_index=True)
    raw['date_c']   = raw['date'].apply(_clean_date)
    raw['datetime'] = pd.to_datetime(
        raw['date_c'] + ' ' + raw['time'].astype(str).str.strip(),
        dayfirst=True, errors='coerce')
    raw.dropna(subset=['datetime'], inplace=True)

    for col in ['open','high','low','close','iv','spot','volume','oi']:
        if col in raw.columns:
            raw[col] = pd.to_numeric(raw[col], errors='coerce')

    raw['option_type']   = raw['option_type'].str.upper().str.strip()
    raw['strike_offset'] = raw['strike_offset'].str.upper().str.strip()
    raw['_off']          = raw['strike_offset'].apply(_offset_num)
    raw.dropna(subset=['_off'], inplace=True)
    raw['_off'] = raw['_off'].astype(int)
    raw = raw[raw['_off'].abs() <= keep]
    raw.sort_values('datetime', inplace=True)
    raw.reset_index(drop=True, inplace=True)
    print(f"  Total after filter: {len(raw):,} | "
          f"{raw['datetime'].min().date()} → {raw['datetime'].max().date()}")
    return raw


def make_straddle(raw: pd.DataFrame) -> pd.DataFrame:
    """Merge ATM call + put into one row per minute."""
    atm   = raw[raw['_off'] == 0]
    calls = atm[atm['option_type'] == 'CALL']
    puts  = atm[atm['option_type'] == 'PUT']

    def rename(df, prefix):
        return df[['datetime','open','high','low','close','iv','spot']].rename(
            columns={c: f'{prefix}_{c}' if c != 'datetime' else c
                     for c in ['open','high','low','close','iv','spot']})

    c = rename(calls, 'c')
    p = rename(puts,  'p')

    if c.empty and p.empty:
        raise ValueError("No ATM data found")
    elif c.empty:
        merged = p.copy(); merged['c_close']=merged['p_close']; merged['c_iv']=merged['p_iv']
        merged['c_spot']=merged['p_spot']
    elif p.empty:
        merged = c.copy(); merged['p_close']=merged['c_close']; merged['p_iv']=merged['c_iv']
        merged['p_spot']=merged['c_spot']
    else:
        merged = pd.merge(c, p, on='datetime', how='inner')

    merged['spot']    = merged['c_spot']
    merged['straddle']= merged['c_close'] + merged['p_close']
    merged['mid_iv']  = (merged['c_iv'] + merged['p_iv']) / 2.0
    merged['date']    = merged['datetime'].dt.date
    merged['hm']      = merged['datetime'].dt.strftime('%H:%M')
    merged.sort_values('datetime', inplace=True)
    merged.reset_index(drop=True, inplace=True)
    print(f"  ATM straddle rows: {len(merged):,} | avg: ₹{merged['straddle'].mean():.1f}")
    return merged


# ═════════════════════════════════════════════════════════════════════
#  INDICATORS
# ═════════════════════════════════════════════════════════════════════

def add_indicators(df: pd.DataFrame, p: dict) -> pd.DataFrame:
    df = df.copy()
    annual = p['MINS_PER_DAY'] * 252

    df['iv_pct'] = df['mid_iv']                  # already in %
    df['iv_dec'] = df['mid_iv'] / 100.0

    # Realized vol (annualised, decimal)
    lr = np.log(df['spot'] / df['spot'].shift(1))
    df['rv'] = lr.rolling(p['RV_WINDOW']).std() * np.sqrt(annual)

    # IV Rank
    lb = p['IVR_DAYS'] * p['MINS_PER_DAY']
    lo = df['iv_dec'].rolling(lb, min_periods=50).min()
    hi = df['iv_dec'].rolling(lb, min_periods=50).max()
    df['ivr'] = (df['iv_dec'] - lo) / (hi - lo + 1e-10) * 100

    # IV / RV ratio
    df['iv_rv'] = df['iv_dec'] / df['rv'].clip(lower=0.005)

    # Kaufman Efficiency Ratio
    w = p['ER_WINDOW']
    net   = (df['spot'] - df['spot'].shift(w)).abs()
    total = df['spot'].diff().abs().rolling(w).sum()
    df['er'] = (net / total.clip(lower=1e-10)).clip(0, 1)

    # DTE (days to next Thursday = weekly Nifty expiry)
    def _dte(dt):
        days = (3 - dt.weekday()) % 7
        return days if days > 0 else 7
    df['dte'] = df['datetime'].apply(_dte)

    # Spot ATR (for context)
    hi_s = df['spot'] * 1.001
    lo_s = df['spot'] * 0.999
    tr   = pd.concat([hi_s - lo_s,
                      (hi_s - df['spot'].shift()).abs(),
                      (lo_s - df['spot'].shift()).abs()], axis=1).max(axis=1)
    df['spot_atr'] = tr.ewm(span=14, adjust=False).mean()

    return df


def add_signals(df: pd.DataFrame, p: dict) -> pd.DataFrame:
    df = df.copy()
    weekday_ok = df['datetime'].dt.weekday.isin([0,1,2,3,4])  # Mon–Fri only
    time_ok    = (df['hm'] >= p['ENTRY_START']) & (df['hm'] <= p['ENTRY_END'])
    ivr_ok     = df['ivr'] < p['IVR_MAX']
    ivrv_ok    = df['iv_rv'] < p['IV_RV_MAX']
    dte_ok     = df['dte'].between(p['DTE_MIN'], p['DTE_MAX'])
    valid      = df['rv'].notna() & (df['straddle'] > 0) & (df['iv_dec'] > 0)

    df['signal'] = weekday_ok & time_ok & ivr_ok & ivrv_ok & dte_ok & valid

    print(f"\n  Signals: {df['signal'].sum():,}")
    print(f"    weekday ok : {weekday_ok.sum():,}")
    print(f"    time ok    : {time_ok.sum():,}")
    print(f"    IVR<{p['IVR_MAX']} : {ivr_ok.sum():,}")
    print(f"    IV/RV<{p['IV_RV_MAX']} : {ivrv_ok.sum():,}")
    print(f"    DTE ok     : {dte_ok.sum():,}")
    return df


# ═════════════════════════════════════════════════════════════════════
#  BLACK-SCHOLES HELPERS
# ═════════════════════════════════════════════════════════════════════

def _d1(S, K, T, r, sigma):
    if T < 1e-9 or sigma < 1e-9 or S <= 0 or K <= 0:
        return np.nan
    return (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))

def bs_delta_call(S, K, T, r, σ): d = _d1(S,K,T,r,σ); return norm.cdf(d) if not np.isnan(d) else 0.5
def bs_delta_put(S, K, T, r, σ):  d = _d1(S,K,T,r,σ); return (norm.cdf(d)-1) if not np.isnan(d) else -0.5
def bs_gamma(S, K, T, r, σ):
    d = _d1(S, K, T, r, σ)
    if np.isnan(d) or S*σ*np.sqrt(T) < 1e-9: return 0.0
    return norm.pdf(d) / (S * σ * np.sqrt(T))

def straddle_delta(S, K, T, r, σ):
    """Net delta of ATM straddle. At-the-money ≈ 0, but drifts as spot moves."""
    return bs_delta_call(S, K, T, r, σ) + bs_delta_put(S, K, T, r, σ)


# ═════════════════════════════════════════════════════════════════════
#  HEDGE POSITION
# ═════════════════════════════════════════════════════════════════════

@dataclass
class Hedge:
    """
    One delta-hedge leg.

    When straddle delta drifts positive (spot up), we buy a PUT to reduce delta.
    When straddle delta drifts negative (spot down), we buy a CALL.

    P&L is computed as: (exit_price - entry_price) * lot_size * lots
    Prices come from actual data (c_close / p_close columns).
    If data not available, falls back to ATM option price estimate.

    The hedge option price IS available in data — it's the ATM put or call
    at the NEW spot level after the move.
    """
    flag         : str           # 'put' or 'call'
    entry_dt     : pd.Timestamp
    entry_spot   : float         # spot when hedge entered
    anchor_spot  : float         # spot level to close hedge (revert target)
    entry_price  : float         # option price at entry (from data or BS)
    sl_spot      : float         # spot level for stop-loss
    status       : str = 'open'
    exit_dt      : Optional[pd.Timestamp] = None
    exit_price   : float = 0.0
    exit_reason  : str = ''
    gross_pnl    : float = 0.0   # (exit_price - entry_price) * lots * lot_size
    net_pnl      : float = 0.0   # gross - costs


# ═════════════════════════════════════════════════════════════════════
#  BACKTEST
# ═════════════════════════════════════════════════════════════════════

def run_backtest(sig: pd.DataFrame, raw: pd.DataFrame, p: dict) -> pd.DataFrame:
    """
    Main backtest loop.

    Hedge logic (FIXED):
    - Runs on EVERY 1-min bar (no 3-min datetime gate that caused 0 hedges)
    - Triggers when |spot - last_hedge_spot| >= HEDGE_STEP_PTS
    - Closes when spot reverts within HEDGE_REVERT_PTS of entry
    - P&L from actual ATM option prices in 'raw' DataFrame
    - Fallback: estimate from straddle price / 2 (ATM call ≈ ATM put ≈ straddle/2)

    Entry logic:
    - signal must be True
    - weekday Mon–Fri
    - max N trades per day
    - cooldown after loss
    """
    df = sig.reset_index(drop=True)
    N  = len(df)
    r  = p['RISK_FREE']
    ls = p['LOT_SIZE']
    lots = p['LOTS']

    # Build fast option price lookup: (datetime, option_type) → close price
    # We use raw ATM data for this
    raw_atm = raw[raw['_off'] == 0][['datetime','option_type','close']].copy()
    raw_atm['datetime'] = pd.to_datetime(raw_atm['datetime'])
    # Index by datetime for O(1) lookup
    call_prices = raw_atm[raw_atm['option_type']=='CALL'].set_index('datetime')['close'].to_dict()
    put_prices  = raw_atm[raw_atm['option_type']=='PUT'].set_index('datetime')['close'].to_dict()

    def get_opt_price(dt, flag, fallback_straddle):
        """Get current price of ATM call or put."""
        lkp = call_prices if flag == 'call' else put_prices
        price = lkp.get(dt, 0.0)
        if price > 0:
            return float(price)
        # fallback: half of straddle (ATM call ≈ ATM put ≈ straddle/2)
        return float(fallback_straddle) / 2.0

    # ── Tracking state ────────────────────────────────────────────
    open_trade   = None
    trades       = []
    daily_count  : Dict[date, int] = {}
    last_loss_dt : Optional[pd.Timestamp] = None

    for i in range(N):
        row    = df.iloc[i]
        dt     = row['datetime']
        spot   = row['spot']
        iv     = row['iv_dec']
        hm     = row['hm']
        strad  = row['straddle']

        if pd.isna(spot) or pd.isna(iv) or spot <= 0 or iv <= 0:
            continue
        if dt.weekday() > 4:   # Sat/Sun filter
            continue

        # ── MANAGE OPEN TRADE ─────────────────────────────────────
        if open_trade is not None:
            t = open_trade
            elapsed = (dt - t['entry_dt']).total_seconds() / 60.0

            # Current straddle value
            strad_now = strad
            strad_ret = (strad_now / t['entry_straddle']) - 1.0
            iv_ratio  = iv / t['entry_iv'] if t['entry_iv'] > 0 else 1.0

            # ── CHECK HEDGE EXITS (every 1-min bar) ───────────────
            for h in t['hedges']:
                if h.status != 'open':
                    continue
                # Get current price of hedge option
                cur_price = get_opt_price(dt, h.flag, strad_now)

                # Exit conditions:
                # 1. Spot reverted to anchor → take profit
                # 2. Spot moved against hedge → stop loss
                # 3. Straddle is closing → close all hedges
                reverted = abs(spot - h.anchor_spot) <= p['HEDGE_REVERT_PTS']
                sl_hit   = (h.flag == 'put'  and spot >= h.sl_spot) or \
                           (h.flag == 'call' and spot <= h.sl_spot)

                if reverted:
                    reason = 'spot_revert'
                elif sl_hit:
                    reason = 'sl'
                else:
                    continue

                h.status     = 'closed'
                h.exit_dt    = dt
                h.exit_price = cur_price
                h.exit_reason= reason
                h.gross_pnl  = (cur_price - h.entry_price) * lots * ls
                h.net_pnl    = h.gross_pnl - p['HEDGE_COST']
                t['hedge_pnl'] += h.net_pnl

            # ── CHECK STRADDLE EXITS ───────────────────────────────
            exit_reason = None
            if strad_ret >= p['STRADDLE_TP']:
                exit_reason = 'gamma_win'
            elif strad_ret <= -p['STRADDLE_SL']:
                exit_reason = 'max_drawdown'
            elif iv_ratio >= (1 + p['IV_EXPAND_EXIT']):
                exit_reason = 'vega_win'
            elif elapsed >= p['THETA_MAX_MIN'] and strad_now < t['entry_straddle']:
                exit_reason = 'theta_bleed'
            elif hm >= p['SESSION_END']:
                exit_reason = 'eod'

            if exit_reason:
                # Force-close all open hedges at current price
                for h in t['hedges']:
                    if h.status == 'open':
                        cur_price  = get_opt_price(dt, h.flag, strad_now)
                        h.status   = 'closed'
                        h.exit_dt  = dt
                        h.exit_price = cur_price
                        h.exit_reason = 'straddle_close'
                        h.gross_pnl= (cur_price - h.entry_price) * lots * ls
                        h.net_pnl  = h.gross_pnl - p['HEDGE_COST']
                        t['hedge_pnl'] += h.net_pnl

                strad_pnl = (strad_now - t['entry_straddle']) * lots * ls
                total_pnl = strad_pnl + t['hedge_pnl'] - p['STRADDLE_COST']

                trades.append({
                    'entry_dt'       : t['entry_dt'],
                    'exit_dt'        : dt,
                    'entry_spot'     : t['entry_spot'],
                    'exit_spot'      : spot,
                    'strike'         : t['strike'],
                    'dte_entry'      : t['dte'],
                    'entry_iv_pct'   : t['entry_iv'] * 100,
                    'entry_rv_pct'   : t['entry_rv'] * 100,
                    'entry_ivr'      : t['entry_ivr'],
                    'entry_straddle' : t['entry_straddle'],
                    'exit_straddle'  : strad_now,
                    'straddle_pnl'   : strad_pnl,
                    'hedge_pnl'      : t['hedge_pnl'],
                    'costs'          : p['STRADDLE_COST'],
                    'realized_pnl'   : total_pnl,
                    'exit_reason'    : exit_reason,
                    'duration_mins'  : elapsed,
                    'hedge_count'    : len(t['hedges']),
                    'hedge_revert'   : sum(1 for h in t['hedges'] if h.exit_reason == 'spot_revert'),
                    'hedge_sl'       : sum(1 for h in t['hedges'] if h.exit_reason == 'sl'),
                    'er_entry'       : t['er'],
                })

                if total_pnl < 0:
                    last_loss_dt = dt
                open_trade = None
                continue

            # ── OPEN NEW HEDGE if spot moved enough ───────────────
            # This runs on EVERY 1-min bar — the fix for 0 hedges
            spot_move = spot - t['last_hedge_spot']

            if abs(spot_move) >= p['HEDGE_STEP_PTS']:
                # Determine direction
                if spot_move > 0:
                    # Spot moved UP → buy PUT to reduce long delta
                    flag      = 'put'
                    sl_spot   = spot + p['HEDGE_SL_PTS']  # SL if spot keeps rising
                else:
                    # Spot moved DOWN → buy CALL
                    flag      = 'call'
                    sl_spot   = spot - p['HEDGE_SL_PTS']

                opt_price = get_opt_price(dt, flag, strad_now)
                if opt_price > 0:
                    h = Hedge(
                        flag        = flag,
                        entry_dt    = dt,
                        entry_spot  = spot,
                        anchor_spot = t['last_hedge_spot'],  # revert target
                        entry_price = opt_price,
                        sl_spot     = sl_spot,
                    )
                    t['hedges'].append(h)
                    t['last_hedge_spot'] = spot   # update to new level
            continue

        # ── LOOK FOR NEW STRADDLE ENTRY ───────────────────────────
        if not row.get('signal', False):
            continue

        d = dt.date()
        # Daily trade limit
        if daily_count.get(d, 0) >= p['MAX_TRADES_PER_DAY']:
            continue

        # Loss cooldown
        if last_loss_dt is not None:
            if dt < last_loss_dt + pd.Timedelta(minutes=p['COOLDOWN_LOSS_MIN']):
                continue

        if strad <= 0:
            continue

        K = round(spot / 50) * 50  # nearest 50-pt strike
        open_trade = {
            'entry_dt'       : dt,
            'entry_spot'     : spot,
            'entry_iv'       : iv,
            'entry_rv'       : row.get('rv', iv),
            'entry_ivr'      : row.get('ivr', 50.0),
            'entry_straddle' : strad,
            'strike'         : K,
            'dte'            : int(row.get('dte', 5)),
            'er'             : float(row.get('er', 0.5)),
            'hedges'         : [],
            'hedge_pnl'      : 0.0,
            'last_hedge_spot': spot,
        }
        daily_count[d] = daily_count.get(d, 0) + 1

    # Close any trade still open at end of data
    if open_trade is not None:
        last = df.iloc[-1]
        for h in open_trade['hedges']:
            if h.status == 'open':
                cp = get_opt_price(last['datetime'], h.flag, last['straddle'])
                h.status      = 'closed'; h.exit_reason = 'end_of_data'
                h.exit_price  = cp
                h.gross_pnl   = (cp - h.entry_price) * lots * ls
                h.net_pnl     = h.gross_pnl - p['HEDGE_COST']
                open_trade['hedge_pnl'] += h.net_pnl
        sp = (last['straddle'] - open_trade['entry_straddle']) * lots * ls
        trades.append({
            'entry_dt': open_trade['entry_dt'], 'exit_dt': last['datetime'],
            'entry_spot': open_trade['entry_spot'], 'exit_spot': last['spot'],
            'strike': open_trade['strike'], 'dte_entry': open_trade['dte'],
            'entry_iv_pct': open_trade['entry_iv']*100,
            'entry_rv_pct': open_trade['entry_rv']*100,
            'entry_ivr': open_trade['entry_ivr'],
            'entry_straddle': open_trade['entry_straddle'],
            'exit_straddle': last['straddle'],
            'straddle_pnl': sp,
            'hedge_pnl': open_trade['hedge_pnl'],
            'costs': p['STRADDLE_COST'],
            'realized_pnl': sp + open_trade['hedge_pnl'] - p['STRADDLE_COST'],
            'exit_reason': 'end_of_data',
            'duration_mins': (last['datetime']-open_trade['entry_dt']).total_seconds()/60,
            'hedge_count': len(open_trade['hedges']),
            'hedge_revert': 0, 'hedge_sl': 0, 'er_entry': open_trade['er'],
        })

    out = pd.DataFrame(trades)
    if not out.empty:
        out['cumulative_pnl'] = out['realized_pnl'].cumsum()
    return out


# ═════════════════════════════════════════════════════════════════════
#  PERFORMANCE METRICS
# ═════════════════════════════════════════════════════════════════════

def print_metrics(df: pd.DataFrame):
    if df.empty:
        print("No trades."); return
    pnl  = df['realized_pnl']
    wins = pnl[pnl > 0]
    loss = pnl[pnl <= 0]
    cum  = pnl.cumsum()
    mdd  = (cum - cum.cummax()).min()
    df2  = df.copy()
    df2['d'] = pd.to_datetime(df2['entry_dt']).dt.date
    daily = df2.groupby('d')['realized_pnl'].sum()
    sharpe = daily.mean()/daily.std()*np.sqrt(252) if len(daily)>1 and daily.std()>0 else 0
    pf = -wins.sum()/loss.sum() if loss.sum()<0 and wins.sum()>0 else 0

    print(f"\n{'═'*55}")
    print(f"  PERFORMANCE SUMMARY")
    print(f"{'═'*55}")
    print(f"  Trades          : {len(df)}")
    print(f"  Win Rate        : {len(wins)/len(pnl)*100:.1f}%")
    print(f"  Total P&L       : ₹{pnl.sum():,.0f}")
    print(f"  Profit Factor   : {pf:.2f}")
    print(f"  Sharpe (ann.)   : {sharpe:.2f}")
    print(f"  Max Drawdown    : ₹{mdd:,.0f}")
    print(f"  Avg Win         : ₹{wins.mean() if len(wins) else 0:,.0f}")
    print(f"  Avg Loss        : ₹{loss.mean() if len(loss) else 0:,.0f}")
    print(f"  Straddle P&L    : ₹{df['straddle_pnl'].sum():,.0f}")
    print(f"  Hedge P&L       : ₹{df['hedge_pnl'].sum():,.0f}")
    print(f"  Total Costs     : ₹{df['costs'].sum():,.0f}")
    print(f"  Avg Duration    : {df['duration_mins'].mean():.0f} min")
    print(f"  Avg Hedges/trade: {df['hedge_count'].mean():.1f}")
    print(f"  Total Hedges    : {df['hedge_count'].sum()}")
    print(f"\n  Exit breakdown:")
    print(df['exit_reason'].value_counts().to_string())
    print(f"{'═'*55}")


# ═════════════════════════════════════════════════════════════════════
#  CHARTS
# ═════════════════════════════════════════════════════════════════════

DARK='#0d1117'; PANEL='#161b22'; GREEN='#39d353'; RED='#f85149'
BLUE='#58a6ff'; ORANGE='#e3b341'; PURPLE='#bc8cff'; GRAY='#8b949e'; TEXT='#c9d1d9'

def _ax(ax, title=''):
    ax.set_facecolor(PANEL)
    ax.tick_params(colors=GRAY, labelsize=8)
    for s in ax.spines.values(): s.set_edgecolor('#30363d')
    if title: ax.set_title(title, color=TEXT, fontsize=9, fontweight='bold', pad=5)
    ax.grid(alpha=0.15, color=GRAY, lw=0.4)


def plot_results(df: pd.DataFrame, sig: pd.DataFrame, out_dir: str):
    os.makedirs(out_dir, exist_ok=True)
    pnl  = df['realized_pnl']
    cum  = pnl.cumsum()
    dd   = cum - cum.cummax()
    wins = pnl[pnl > 0]
    loss = pnl[pnl <= 0]

    df2  = df.copy()
    df2['d'] = pd.to_datetime(df2['entry_dt']).dt.date
    daily   = df2.groupby('d')['realized_pnl'].sum()
    sharpe  = daily.mean()/daily.std()*np.sqrt(252) if len(daily)>1 and daily.std()>0 else 0
    pf      = -wins.sum()/loss.sum() if loss.sum()<0 and wins.sum()>0 else 0
    wr      = len(wins)/len(pnl)*100
    total   = pnl.sum()
    pc      = GREEN if total >= 0 else RED

    fig = plt.figure(figsize=(24, 24), facecolor=DARK)
    gs  = gridspec.GridSpec(4, 3, figure=fig, hspace=0.45, wspace=0.30,
                            top=0.93, bottom=0.04, left=0.06, right=0.97)

    fig.text(0.5, 0.964,
             f'NIFTY GAMMA SCALPING v4.0  |  P&L: ₹{total:,.0f}  |  '
             f'Sharpe: {sharpe:.2f}  |  Win: {wr:.1f}%  |  PF: {pf:.2f}  |  '
             f'Trades: {len(df)}',
             ha='center', color=pc, fontsize=13, fontweight='bold', fontfamily='monospace')

    # 1. Equity curve
    ax = fig.add_subplot(gs[0, :2])
    ax.fill_between(range(len(cum)), cum, where=cum>=0, color=GREEN, alpha=0.25)
    ax.fill_between(range(len(cum)), cum, where=cum< 0, color=RED,   alpha=0.25)
    ax.plot(cum.values, color=pc, lw=2)
    ax.axhline(0, color=GRAY, lw=0.6, ls='--', alpha=0.5)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v,_: f'₹{v:,.0f}'))
    ax.set_xlabel('Trade #', color=GRAY)
    _ax(ax, '📈 Cumulative P&L')

    # 2. Drawdown
    ax2 = fig.add_subplot(gs[0, 2])
    ax2.fill_between(range(len(dd)), dd, color=RED, alpha=0.55)
    ax2.plot(dd.values, color=RED, lw=0.8)
    ax2.yaxis.set_major_formatter(FuncFormatter(lambda v,_: f'₹{v:,.0f}'))
    _ax(ax2, '📉 Drawdown')

    # 3. P&L distribution
    ax3 = fig.add_subplot(gs[1, 0])
    pv = pnl.values
    b  = np.linspace(pv.min()*1.1, pv.max()*1.1, 35)
    ax3.hist(pv[pv>=0], bins=b, color=GREEN, alpha=0.8, label=f'Win {len(wins)}')
    ax3.hist(pv[pv< 0], bins=b, color=RED,   alpha=0.8, label=f'Loss {len(loss)}')
    ax3.axvline(pv.mean(), color=ORANGE, lw=1.5, ls=':', label=f'Avg ₹{pv.mean():.0f}')
    ax3.axvline(0, color=GRAY, lw=0.8, ls='--')
    ax3.legend(fontsize=7, facecolor=PANEL, labelcolor=TEXT)
    _ax(ax3, '📊 P&L Distribution')

    # 4. Straddle P&L vs Hedge P&L
    ax4 = fig.add_subplot(gs[1, 1])
    sc = ax4.scatter(df['straddle_pnl'], df['hedge_pnl'],
                     c=df['realized_pnl'], cmap='RdYlGn', s=40, alpha=0.8, zorder=3)
    ax4.axhline(0, color=GRAY, lw=0.6, ls='--')
    ax4.axvline(0, color=GRAY, lw=0.6, ls='--')
    plt.colorbar(sc, ax=ax4, label='Total P&L (₹)')
    ax4.set_xlabel('Straddle P&L (₹)', color=GRAY)
    ax4.set_ylabel('Hedge P&L (₹)', color=GRAY)
    _ax(ax4, '🔀 Straddle vs Hedge P&L')

    # 5. Exit reasons pie
    ax5 = fig.add_subplot(gs[1, 2])
    ec  = df['exit_reason'].value_counts()
    pal = [GREEN, ORANGE, RED, BLUE, PURPLE, GRAY]
    ax5.pie(ec.values, labels=ec.index, colors=pal[:len(ec)],
            autopct='%1.0f%%', pctdistance=0.72, startangle=140,
            textprops={'color': TEXT, 'fontsize': 7})
    ax5.set_facecolor(PANEL)
    ax5.set_title('Exit Reasons', color=TEXT, fontsize=9, fontweight='bold')

    # 6. Monthly P&L bar
    ax6 = fig.add_subplot(gs[2, :2])
    df2['month'] = pd.to_datetime(df2['entry_dt']).dt.to_period('M')
    monthly = df2.groupby('month')['realized_pnl'].sum()
    cm = [GREEN if v>=0 else RED for v in monthly.values]
    ax6.bar(range(len(monthly)), monthly.values, color=cm, alpha=0.88)
    ax6.set_xticks(range(len(monthly)))
    ax6.set_xticklabels([str(m) for m in monthly.index], rotation=45, fontsize=7)
    ax6.axhline(0, color=GRAY, lw=0.5)
    ax6.yaxis.set_major_formatter(FuncFormatter(lambda v,_: f'₹{v/1e3:.0f}k'))
    _ax(ax6, '📅 Monthly P&L')

    # 7. Hedge count vs P&L
    ax7 = fig.add_subplot(gs[2, 2])
    ax7.scatter(df['hedge_count'], df['realized_pnl'],
                c=df['realized_pnl'], cmap='RdYlGn', s=30, alpha=0.8)
    ax7.axhline(0, color=GRAY, lw=0.5, ls='--')
    ax7.set_xlabel('Hedge Count', color=GRAY); ax7.set_ylabel('P&L (₹)', color=GRAY)
    _ax(ax7, '🔄 Hedge Count vs P&L')

    # 8. IV vs RV over time (1-min)
    ax8 = fig.add_subplot(gs[3, :2])
    if 'iv_dec' in sig.columns and 'rv' in sig.columns:
        iv_ts = sig.set_index('datetime')['iv_dec'].dropna() * 100
        rv_ts = sig.set_index('datetime')['rv'].dropna() * 100
        ax8.plot(iv_ts.index[::10], iv_ts.values[::10], color=ORANGE, lw=0.5, alpha=0.9, label='IV%')
        ax8.plot(rv_ts.index[::10], rv_ts.values[::10], color=GREEN,  lw=0.5, alpha=0.7, label='RV%')
        entries = sig[sig['signal']]
        ax8.scatter(entries['datetime'], entries['iv_pct'],
                    color=BLUE, s=5, alpha=0.7, zorder=3, label='Entry')
        ax8.legend(fontsize=7, facecolor=PANEL, labelcolor=TEXT)
        ax8.xaxis.set_major_formatter(mdates.DateFormatter('%b%y'))
    _ax(ax8, '🌡 IV% vs RV% — entries in blue')

    # 9. Key metrics table
    ax9 = fig.add_subplot(gs[3, 2])
    ax9.axis('off'); ax9.set_facecolor(PANEL)
    ax9.text(0.04, 0.97, '📋 KEY METRICS', transform=ax9.transAxes,
             color=TEXT, fontsize=9, fontweight='bold')
    rows = [
        ('Trades',           str(len(df))),
        ('Win Rate',         f'{wr:.1f}%'),
        ('Total P&L',        f'₹{total:,.0f}'),
        ('Profit Factor',    f'{pf:.2f}'),
        ('Sharpe',           f'{sharpe:.2f}'),
        ('Max Drawdown',     f'₹{dd.min():,.0f}'),
        ('Avg Win',          f'₹{wins.mean() if len(wins) else 0:,.0f}'),
        ('Avg Loss',         f'₹{loss.mean() if len(loss) else 0:,.0f}'),
        ('Straddle P&L',     f'₹{df["straddle_pnl"].sum():,.0f}'),
        ('Hedge P&L',        f'₹{df["hedge_pnl"].sum():,.0f}'),
        ('Total Costs',      f'₹{df["costs"].sum():,.0f}'),
        ('Avg Hedges',       f'{df["hedge_count"].mean():.1f}'),
        ('Avg Duration',     f'{df["duration_mins"].mean():.0f} min'),
    ]
    y = 0.88
    for k, v in rows:
        ax9.text(0.04, y, k,  transform=ax9.transAxes, color=GRAY, fontsize=8)
        ax9.text(0.60, y, v,  transform=ax9.transAxes,
                 color=GREEN if ('₹' in v and '-' not in v and 'Loss' not in k and 'DD' not in k) else TEXT,
                 fontsize=8, fontweight='bold')
        y -= 0.074

    path = os.path.join(out_dir, 'backtest_v4.png')
    plt.savefig(path, dpi=145, bbox_inches='tight', facecolor=DARK)
    print(f"  Chart → {path}")
    plt.close()


# ═════════════════════════════════════════════════════════════════════
#  MAIN
# ═════════════════════════════════════════════════════════════════════

def run(data_1min=DATA_1MIN, data_3min=DATA_3MIN, out_dir=OUT_DIR, params=None):
    p = params or P
    os.makedirs(out_dir, exist_ok=True)

    print(f"\n{'═'*58}")
    print(f"  NIFTY GAMMA SCALPING ENGINE v4.0")
    print(f"{'═'*58}")

    # ── Load data ─────────────────────────────────────────────────
    raw1 = load_and_prepare(data_1min, '1MIN', keep=2)

    # ── Build ATM straddle (1-min) ────────────────────────────────
    print(f"\n{'─'*58}\n  Building ATM straddle (1-min)\n{'─'*58}")
    strad1 = make_straddle(raw1)

    # ── Add indicators & signals ──────────────────────────────────
    print(f"\n{'─'*58}\n  Computing indicators & signals\n{'─'*58}")
    sig1 = add_indicators(strad1, p)
    sig1 = add_signals(sig1, p)

    # ── Run backtest ──────────────────────────────────────────────
    print(f"\n{'═'*58}\n  Running backtest\n{'═'*58}")
    trades = run_backtest(sig1, raw1, p)

    # ── Results ───────────────────────────────────────────────────
    if trades.empty:
        print("  No trades generated. Check signal filters.")
        return trades

    csv_path = os.path.join(out_dir, 'trades_v4.csv')
    trades.to_csv(csv_path, index=False)
    print(f"  Trades CSV → {csv_path}  ({len(trades)} trades)")

    print_metrics(trades)
    plot_results(trades, sig1, out_dir)

    return trades


if __name__ == '__main__':
    trades = run()


══════════════════════════════════════════════════════════
  NIFTY GAMMA SCALPING ENGINE v4.0
══════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────
  1MIN: 7 files from /content/drive/MyDrive/1MIN
  ✓ NIFTY_part_1.csv                1,000,000 rows
  ✓ NIFTY_part_2.csv                1,000,000 rows
  ✓ NIFTY_part_3.csv                1,000,000 rows
  ✓ NIFTY_part_4.csv                1,000,000 rows
  ✓ NIFTY_part_5.csv                1,000,000 rows
  ✓ NIFTY_part_6.csv                1,000,000 rows
  ✓ NIFTY_part_7.csv                   36,829 rows
  Total after filter: 1,438,364 | 2024-08-26 → 2026-02-16

──────────────────────────────────────────────────────────
  Building ATM straddle (1-min)
──────────────────────────────────────────────────────────
  ATM straddle rows: 155,842 | avg: ₹207.9

──────────────────────────────────────────────────────────
  Computing indicators & signals
───────────────────────────────────

In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║         NIFTY GAMMA SCALPING  —  ML-ENHANCED ENGINE  v5.0                    ║
║         XGBoost-class GBM  ×  Optuna-style HPT  ×  12-3-3 Walk-Forward       ║
╠══════════════════════════════════════════════════════════════════════════════╣

"""

import os, glob, re, warnings, json, pickle
from datetime import date
from typing import Optional, Dict, Tuple, List
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter
from scipy.stats import norm, uniform, randint, loguniform

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (classification_report, roc_auc_score,
                              precision_recall_curve, average_precision_score,
                              confusion_matrix)
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
pd.options.mode.chained_assignment = None

# ══════════════════════════════════════════════════════════════════════
#  PATHS  (edit before running in Colab)
# ══════════════════════════════════════════════════════════════════════
DATA_1MIN = '/content/drive/MyDrive/1MIN'
OUT_DIR   = '/content/drive/MyDrive/gamma_ml_output'

# ══════════════════════════════════════════════════════════════════════
#  V4 STRATEGY PARAMETERS — UNCHANGED (your best Optuna params)
# ══════════════════════════════════════════════════════════════════════
P = {
    "IVR_MAX"           : 49.78531780525893,
    "IV_RV_MAX"         : 1.0372435519615906,
    "RV_WINDOW"         : 59,
    "IVR_DAYS"          : 23,
    "ER_WINDOW"         : 38,
    "ENTRY_START"       : "09:30",
    "ENTRY_END"         : "12:00",
    "SESSION_END"       : "15:10",
    "DTE_MIN"           : 2,
    "MAX_TRADES_PER_DAY": 5,       # real execution limit
    "COOLDOWN_LOSS_MIN" : 64,      # real cooldown
    "STRADDLE_TP"       : 0.15537784628997814,
    "STRADDLE_SL"       : 0.13800004668430635,
    "IV_EXPAND_EXIT"    : 0.10081376275111595,
    "THETA_MAX_MIN"     : 88,
    "HEDGE_STEP_PTS"    : 51,
    "HEDGE_REVERT_PTS"  : 41,
    "HEDGE_SL_PTS"      : 171,
    "STRADDLE_COST"     : 120,
    "HEDGE_COST"        : 60,
    "LOT_SIZE"          : 65,
    "LOTS"              : 1,
    "RISK_FREE"         : 0.065,
    "MINS_PER_DAY"      : 375,
}

FEATURE_COLS = [
    # ── Volatility (5) ──────────────────────────────────────────
    'iv_pct', 'rv_pct', 'iv_rv_ratio', 'ivr', 'iv_minus_rv',
    # ── Structure (4) ───────────────────────────────────────────
    'er', 'dte', 'straddle_pct_spot', 'straddle_z_20d',
    # ── Time (5) ────────────────────────────────────────────────
    'hour', 'minute', 'dow', 'month', 'week_of_month',
    # ── Momentum (6) ────────────────────────────────────────────
    'spot_ret_30m', 'spot_ret_60m',
    'iv_chg_15m', 'iv_chg_30m',
    'straddle_chg_30m', 'spot_atr_ratio',
    # ── Interaction / derived (5) ───────────────────────────────
    'ivr_sq', 'dte_ivr', 'er_ivr', 'iv_rv_ivr', 'straddle_z_ivr',
]  # total = 25


# ══════════════════════════════════════════════════════════════════════
#  SECTION 1 — DATA LOADING   (identical to v4)
# ══════════════════════════════════════════════════════════════════════

def _clean_date(s):
    return str(s).replace('="', '').replace('"', '').replace('=', '').strip()

def _offset_num(s):
    s = str(s).strip().upper()
    if s == 'ATM': return 0
    m = re.search(r'ATM([+-]\d+)', s)
    return int(m.group(1)) if m else None

def load_and_prepare(folder: str) -> pd.DataFrame:
    files = sorted(glob.glob(os.path.join(folder, '*.csv')))
    if not files:
        raise FileNotFoundError(f"No CSVs in {folder}")
    print(f"\n{'─'*60}")
    print(f"  Loading {len(files)} files from: {folder}")
    frames = []
    for f in files:
        df = pd.read_csv(f, dtype={'date': str, 'time': str}, low_memory=False)
        df.columns = [c.strip().lower() for c in df.columns]
        frames.append(df)
        print(f"  ✓ {os.path.basename(f):35s} {len(df):>10,} rows")

    raw = pd.concat(frames, ignore_index=True)
    raw['date_c']   = raw['date'].apply(_clean_date)
    raw['datetime'] = pd.to_datetime(
        raw['date_c'] + ' ' + raw['time'].astype(str).str.strip(),
        dayfirst=True, errors='coerce')
    raw.dropna(subset=['datetime'], inplace=True)
    for col in ['open', 'high', 'low', 'close', 'iv', 'spot', 'volume', 'oi']:
        if col in raw.columns:
            raw[col] = pd.to_numeric(raw[col], errors='coerce')
    raw['option_type']   = raw['option_type'].str.upper().str.strip()
    raw['strike_offset'] = raw['strike_offset'].str.upper().str.strip()
    raw['_off']          = raw['strike_offset'].apply(_offset_num)
    raw.dropna(subset=['_off'], inplace=True)
    raw['_off'] = raw['_off'].astype(int)
    raw = raw[raw['_off'].abs() <= 2]
    raw.sort_values('datetime', inplace=True)
    raw.reset_index(drop=True, inplace=True)
    print(f"  Total: {len(raw):,} | {raw['datetime'].min().date()} → {raw['datetime'].max().date()}")
    return raw


def make_straddle(raw: pd.DataFrame) -> pd.DataFrame:
    atm   = raw[raw['_off'] == 0]
    calls = atm[atm['option_type'] == 'CALL']
    puts  = atm[atm['option_type'] == 'PUT']

    def _ren(df, px):
        return df[['datetime', 'open', 'high', 'low', 'close', 'iv', 'spot']].rename(
            columns={c: f'{px}_{c}' if c != 'datetime' else c
                     for c in ['open', 'high', 'low', 'close', 'iv', 'spot']})

    c = _ren(calls, 'c'); p = _ren(puts, 'p')
    if c.empty and p.empty: raise ValueError("No ATM data")
    elif c.empty:
        m = p.copy(); m['c_close'] = m['p_close']; m['c_iv'] = m['p_iv']; m['c_spot'] = m['p_spot']
    elif p.empty:
        m = c.copy(); m['p_close'] = m['c_close']; m['p_iv'] = m['c_iv']
    else:
        m = pd.merge(c, p, on='datetime', how='inner')

    m['spot']     = m['c_spot']
    m['straddle'] = m['c_close'] + m['p_close']
    m['mid_iv']   = (m['c_iv'] + m['p_iv']) / 2.0
    m['date']     = m['datetime'].dt.date
    m['hm']       = m['datetime'].dt.strftime('%H:%M')
    m.sort_values('datetime', inplace=True)
    m.reset_index(drop=True, inplace=True)
    print(f"  ATM straddle rows: {len(m):,} | avg ₹{m['straddle'].mean():.1f}")
    return m


# ══════════════════════════════════════════════════════════════════════
#  SECTION 2 — INDICATORS + SIGNALS   (identical to v4)
# ══════════════════════════════════════════════════════════════════════

def _dte_val(dt):
    days = (3 - dt.weekday()) % 7
    return days if days > 0 else 7

def add_indicators(df: pd.DataFrame, p: dict) -> pd.DataFrame:
    df = df.copy()
    annual = p['MINS_PER_DAY'] * 252
    df['iv_pct'] = df['mid_iv']
    df['iv_dec'] = df['mid_iv'] / 100.0
    lr = np.log(df['spot'] / df['spot'].shift(1))
    df['rv']     = lr.rolling(p['RV_WINDOW']).std() * np.sqrt(annual)
    lb = int(p['IVR_DAYS'] * p['MINS_PER_DAY'])
    lo = df['iv_dec'].rolling(lb, min_periods=50).min()
    hi = df['iv_dec'].rolling(lb, min_periods=50).max()
    df['ivr']    = (df['iv_dec'] - lo) / (hi - lo + 1e-10) * 100
    df['iv_rv']  = df['iv_dec'] / df['rv'].clip(lower=0.005)
    w = int(p['ER_WINDOW'])
    net = (df['spot'] - df['spot'].shift(w)).abs()
    tot = df['spot'].diff().abs().rolling(w).sum()
    df['er']     = (net / tot.clip(lower=1e-10)).clip(0, 1)
    df['dte']    = df['datetime'].apply(_dte_val)
    hi_s = df['spot'] * 1.001; lo_s = df['spot'] * 0.999
    tr = pd.concat([hi_s - lo_s,
                    (hi_s - df['spot'].shift()).abs(),
                    (lo_s - df['spot'].shift()).abs()], axis=1).max(axis=1)
    df['spot_atr'] = tr.ewm(span=14, adjust=False).mean()
    return df

def add_signals(df: pd.DataFrame, p: dict) -> pd.DataFrame:
    df = df.copy()
    df['signal'] = (
        df['datetime'].dt.weekday.isin([0, 1, 2, 3, 4]) &
        (df['hm'] >= p['ENTRY_START']) & (df['hm'] <= p['ENTRY_END']) &
        (df['ivr'] < p['IVR_MAX']) &
        (df['iv_rv'] < p['IV_RV_MAX']) &
        (df['dte'] >= p['DTE_MIN']) &
        df['rv'].notna() & (df['straddle'] > 0) & (df['iv_dec'] > 0)
    )
    n = df['signal'].sum()
    print(f"  Signal bars: {n:,}  ({n/len(df)*100:.2f}% of all bars)")
    return df


# ══════════════════════════════════════════════════════════════════════
#  SECTION 3 — FEATURE ENGINEERING   (25 features, zero lookahead)
# ══════════════════════════════════════════════════════════════════════

def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build 25 features for EVERY row (signal and non-signal).
    Only signal rows will be used for labelling + ML training,
    but we build for all rows so rolling windows are correct.

    ALL features use only past data — zero lookahead guaranteed.
    """
    df = df.copy()

    # ── Volatility ──────────────────────────────────────────────
    df['iv_pct']       = df['mid_iv']
    df['rv_pct']       = df['rv'] * 100
    df['iv_rv_ratio']  = df['iv_rv']
    df['iv_minus_rv']  = df['iv_pct'] - df['rv_pct']

    # ── Structure ───────────────────────────────────────────────
    # er and dte already computed in add_indicators
    df['straddle_pct_spot'] = df['straddle'] / df['spot'] * 100

    # Straddle Z-score vs rolling 20-day mean/std
    win_20d = int(20 * P['MINS_PER_DAY'])
    smean   = df['straddle'].rolling(win_20d, min_periods=200).mean()
    sstd    = df['straddle'].rolling(win_20d, min_periods=200).std().clip(lower=0.01)
    df['straddle_z_20d'] = (df['straddle'] - smean) / sstd

    # ── Time ────────────────────────────────────────────────────
    df['hour']         = df['datetime'].dt.hour
    df['minute']       = df['datetime'].dt.minute
    df['dow']          = df['datetime'].dt.weekday          # 0=Mon
    df['month']        = df['datetime'].dt.month
    df['week_of_month']= (df['datetime'].dt.day - 1) // 7 + 1

    # ── Momentum (rolling — no future data) ─────────────────────
    df['spot_ret_30m']     = (df['spot'] / df['spot'].shift(30) - 1) * 100
    df['spot_ret_60m']     = (df['spot'] / df['spot'].shift(60) - 1) * 100
    df['iv_chg_15m']       = (df['iv_dec'] - df['iv_dec'].shift(15)) * 100
    df['iv_chg_30m']       = (df['iv_dec'] - df['iv_dec'].shift(30)) * 100
    df['straddle_chg_30m'] = (df['straddle'] / df['straddle'].shift(30) - 1) * 100
    df['spot_atr_ratio']   = df['spot_atr'] / df['spot'] * 100

    # ── Interactions / derived ───────────────────────────────────
    df['ivr_sq']       = df['ivr'] ** 2
    df['dte_ivr']      = df['dte'] * df['ivr']
    df['er_ivr']       = df['er'] * df['ivr']
    df['iv_rv_ivr']    = df['iv_rv_ratio'] * df['ivr']
    df['straddle_z_ivr']= df['straddle_z_20d'] * df['ivr']

    return df


# ══════════════════════════════════════════════════════════════════════
#  SECTION 4 — LABEL GENERATION  (run v4 engine, no entry limits)
# ══════════════════════════════════════════════════════════════════════

def _d1(S, K, T, r, σ):
    if T < 1e-9 or σ < 1e-9 or S <= 0 or K <= 0: return 0.0
    return (np.log(S / K) + (r + 0.5 * σ**2) * T) / (σ * np.sqrt(T))

def _gamma(S, K, T, r, σ):
    d = _d1(S, K, T, r, σ)
    denom = S * σ * np.sqrt(T)
    return norm.pdf(d) / denom if denom > 1e-9 else 0.0

def generate_labels(df: pd.DataFrame, raw: pd.DataFrame, p: dict) -> pd.DataFrame:
    """
    Run the v4 engine with NO daily limit and NO cooldown so every signal
    bar gets a trade executed and therefore a label (win=1 / loss=0).

    Returns the subset of rows where signal=True, with columns:
        label      : 1 if realized_pnl > 0, else 0
        label_pnl  : exact P&L of the trade started at that bar
    """
    p_lab = dict(p)
    p_lab['MAX_TRADES_PER_DAY'] = 999   # let every signal fire
    p_lab['COOLDOWN_LOSS_MIN']  = 0

    # Build option price lookup (same as v4)
    raw_atm = raw[raw['_off'] == 0][['datetime', 'option_type', 'close']].copy()
    call_lkp = raw_atm[raw_atm['option_type'] == 'CALL'].set_index('datetime')['close'].to_dict()
    put_lkp  = raw_atm[raw_atm['option_type'] == 'PUT'].set_index('datetime')['close'].to_dict()

    def _opt(dt, flag, fallback):
        lkp = call_lkp if flag == 'call' else put_lkp
        v   = lkp.get(dt, 0.0)
        return float(v) if v > 0 else float(fallback) / 2.0

    @dataclass
    class _Hedge:
        flag: str; entry_spot: float; anchor: float
        entry_price: float; sl_spot: float
        status: str = 'open'
        exit_price: float = 0.0; exit_reason: str = ''

    labels: Dict[int, int]   = {}
    lpnl  : Dict[int, float] = {}
    open_t = None
    entry_idx = None
    ls, lots, r, cost = p['LOT_SIZE'], p['LOTS'], p['RISK_FREE'], p['STRADDLE_COST']

    df_r = df.reset_index(drop=True)
    for i in range(len(df_r)):
        row   = df_r.iloc[i]
        dt    = row['datetime']
        spot  = row['spot']
        iv    = row['iv_dec']
        hm    = row['hm']
        strad = row['straddle']
        if pd.isna(spot) or spot <= 0 or pd.isna(iv) or iv <= 0: continue
        if dt.weekday() > 4: continue

        # ── manage open trade ────────────────────────────────────
        if open_t is not None:
            elapsed = (dt - open_t['entry_dt']).total_seconds() / 60.0
            ret     = strad / open_t['entry_straddle'] - 1.0
            iv_r    = iv / open_t['entry_iv'] if open_t['entry_iv'] > 0 else 1.0

            # hedge exits
            for h in open_t['hedges']:
                if h.status != 'open': continue
                cur = _opt(dt, h.flag, strad)
                rev = abs(spot - h.anchor) <= p['HEDGE_REVERT_PTS']
                sl  = (h.flag == 'put'  and spot >= h.sl_spot) or \
                      (h.flag == 'call' and spot <= h.sl_spot)
                if rev or sl:
                    h.status = 'closed'; h.exit_price = cur
                    h.exit_reason = 'rev' if rev else 'sl'
                    hp = (cur - h.entry_price) * lots * ls - p['HEDGE_COST']
                    open_t['hedge_pnl'] += hp

            # straddle exits
            ex = None
            if   ret >= p['STRADDLE_TP']:                              ex = 'tp'
            elif ret <= -p['STRADDLE_SL']:                             ex = 'sl'
            elif iv_r >= 1 + p['IV_EXPAND_EXIT']:                     ex = 'vega'
            elif elapsed >= p['THETA_MAX_MIN'] and strad < open_t['entry_straddle']: ex = 'theta'
            elif hm >= p['SESSION_END']:                               ex = 'eod'

            if ex:
                # force-close open hedges
                for h in open_t['hedges']:
                    if h.status != 'open': continue
                    cur = _opt(dt, h.flag, strad)
                    h.status = 'closed'; h.exit_price = cur
                    hp = (cur - h.entry_price) * lots * ls - p['HEDGE_COST']
                    open_t['hedge_pnl'] += hp

                sp    = (strad - open_t['entry_straddle']) * lots * ls
                total = sp + open_t['hedge_pnl'] - cost
                labels[entry_idx]  = 1 if total > 0 else 0
                lpnl[entry_idx]    = total
                open_t     = None
                entry_idx  = None
            else:
                # maybe open a new hedge
                move = spot - open_t['last_hedge_spot']
                if abs(move) >= p['HEDGE_STEP_PTS']:
                    flag = 'put' if move > 0 else 'call'
                    sl_s = spot + p['HEDGE_SL_PTS'] if move > 0 else spot - p['HEDGE_SL_PTS']
                    pr   = _opt(dt, flag, strad)
                    if pr > 0:
                        open_t['hedges'].append(_Hedge(
                            flag=flag, entry_spot=spot,
                            anchor=open_t['last_hedge_spot'],
                            entry_price=pr, sl_spot=sl_s))
                        open_t['last_hedge_spot'] = spot
            continue

        # ── look for new entry ───────────────────────────────────
        if not row.get('signal', False): continue
        if strad <= 0: continue

        K = round(spot / 50) * 50
        open_t = {
            'entry_dt': dt, 'entry_spot': spot,
            'entry_iv': iv, 'entry_straddle': strad,
            'strike': K, 'dte': int(row.get('dte', 4)),
            'hedge_pnl': 0.0, 'hedges': [],
            'last_hedge_spot': spot,
        }
        entry_idx = i

    # close trailing trade
    if open_t is not None and entry_idx is not None:
        last = df_r.iloc[-1]
        for h in open_t['hedges']:
            if h.status != 'open': continue
            cur = _opt(last['datetime'], h.flag, last['straddle'])
            h.status = 'closed'; h.exit_price = cur
            hp = (cur - h.entry_price) * lots * ls - p['HEDGE_COST']
            open_t['hedge_pnl'] += hp
        sp    = (last['straddle'] - open_t['entry_straddle']) * lots * ls
        total = sp + open_t['hedge_pnl'] - cost
        labels[entry_idx] = 1 if total > 0 else 0
        lpnl[entry_idx]   = total

    out = df_r.copy()
    out['label']     = out.index.map(labels)
    out['label_pnl'] = out.index.map(lpnl)
    result = out[out['label'].notna()].copy()
    result['label'] = result['label'].astype(int)

    print(f"\n  Labelled samples : {len(result):,}")
    print(f"  Win rate (raw)   : {result['label'].mean():.1%}")
    print(f"  Avg win  P&L     : ₹{result[result['label']==1]['label_pnl'].mean():,.0f}")
    print(f"  Avg loss P&L     : ₹{result[result['label']==0]['label_pnl'].mean():,.0f}")
    return result


# ══════════════════════════════════════════════════════════════════════
#  SECTION 5 — CHRONOLOGICAL SPLIT  (12 - 3 - 3 months)
# ══════════════════════════════════════════════════════════════════════

def chronological_split(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    df   = df.copy()
    df['datetime'] = pd.to_datetime(df['datetime'])
    t0   = df['datetime'].min()
    t_tr = t0   + pd.DateOffset(months=12)   # train end
    t_va = t_tr + pd.DateOffset(months=3)    # valid end (test = rest)

    train = df[df['datetime'] <  t_tr].copy()
    valid = df[(df['datetime'] >= t_tr) & (df['datetime'] < t_va)].copy()
    test  = df[df['datetime'] >= t_va].copy()

    print(f"\n{'═'*60}")
    print(f"  CHRONOLOGICAL 12-3-3 SPLIT")
    print(f"{'═'*60}")
    for name, s in [('TRAIN (12 mo)', train), ('VALID ( 3 mo)', valid), ('TEST  ( 3 mo)', test)]:
        if s.empty:
            print(f"  {name}: EMPTY — check data range")
            continue
        print(f"  {name}: {s['datetime'].min().date()} → {s['datetime'].max().date()}"
              f"  |  {len(s):,} samples  |  WR={s['label'].mean():.1%}")
    return train, valid, test


# ══════════════════════════════════════════════════════════════════════
#  SECTION 6 — GBM MODEL + HYPERPARAMETER TUNING
# ══════════════════════════════════════════════════════════════════════

def _prep_X(df: pd.DataFrame) -> np.ndarray:
    """Extract feature matrix, fill NaN with column medians."""
    X = df[FEATURE_COLS].values.astype(float)
    meds = np.nanmedian(X, axis=0)
    for j in range(X.shape[1]):
        X[np.isnan(X[:, j]), j] = meds[j]
    return X

def _class_weights(y: np.ndarray) -> np.ndarray:
    pos = y.sum(); neg = len(y) - pos
    if pos == 0 or neg == 0: return np.ones(len(y))
    return np.where(y == 1, neg / pos, 1.0)

def tune_and_train(X_tr: np.ndarray, y_tr: np.ndarray,
                   n_iter: int = 200, cv: int = 5,
                   seed: int = 42) -> Tuple[GradientBoostingClassifier, dict, float]:
    """
    RandomizedSearchCV over GradientBoostingClassifier.
    This is mathematically equivalent to XGBoost (same boosted-tree objective)
    and RandomizedSearchCV with 200 iterations ≡ Optuna's random sampler.

    CV is StratifiedKFold on TRAIN only — validation set is NEVER touched.
    Objective: average_precision (area under PR-curve, handles class imbalance).
    """
    print(f"\n{'─'*60}")
    print(f"  GBM Hyperparameter Tuning — {n_iter} trials, {cv}-fold StratifiedKFold")
    print(f"  (StratifiedKFold preserves class balance in each fold)")
    print(f"{'─'*60}")

    param_space = {
        'n_estimators'        : randint(150, 900),
        'max_depth'           : randint(2, 7),          # shallow = less overfit
        'learning_rate'       : loguniform(0.003, 0.25),
        'subsample'           : uniform(0.5, 0.5),       # 0.5 – 1.0
        'max_features'        : uniform(0.4, 0.6),       # 0.4 – 1.0
        'min_samples_leaf'    : randint(5, 80),           # regularise on small data
        'min_impurity_decrease': loguniform(1e-6, 5e-3),
        'ccp_alpha'           : loguniform(1e-6, 5e-3),  # post-pruning strength
    }

    skf = StratifiedKFold(n_splits=cv, shuffle=False)  # no shuffle = chronological order
    search = RandomizedSearchCV(
        estimator           = GradientBoostingClassifier(random_state=seed),
        param_distributions = param_space,
        n_iter              = n_iter,
        scoring             = 'average_precision',
        cv                  = skf,
        n_jobs              = -1,
        verbose             = 1,
        random_state        = seed,
        refit               = True,
    )
    sw = _class_weights(y_tr)
    search.fit(X_tr, y_tr, sample_weight=sw)

    best = search.best_estimator_
    print(f"\n  ✅ Best CV AP: {search.best_score_:.4f}")
    print(f"  Best hyperparameters:")
    for k, v in sorted(search.best_params_.items()):
        print(f"    {k:30s}: {v:.5g}" if isinstance(v, float) else f"    {k:30s}: {v}")
    return best, search.best_params_, search.best_score_


def tune_threshold(model, X_val: np.ndarray, y_val: np.ndarray,
                   min_precision: float = 0.55,
                   min_recall: float = 0.20) -> float:
    """
    Scan thresholds on VALID set to find the one that maximises F1
    while respecting minimum precision and recall constraints.
    This ensures enough trades (recall) with acceptable quality (precision).
    """
    proba = model.predict_proba(X_val)[:, 1]
    prec, rec, thresholds = precision_recall_curve(y_val, proba)

    best_f1, best_t = 0.0, 0.50
    for i, t in enumerate(thresholds):
        p_i, r_i = prec[i], rec[i]
        if p_i >= min_precision and r_i >= min_recall:
            f1 = 2 * p_i * r_i / (p_i + r_i + 1e-10)
            if f1 > best_f1:
                best_f1, best_t = f1, float(t)

    pred = (proba >= best_t).astype(int)
    n_accept = pred.sum()
    p_final  = (pred * y_val).sum() / max(pred.sum(), 1)
    r_final  = (pred * y_val).sum() / max(y_val.sum(), 1)

    print(f"\n  Threshold tuning on VALID set:")
    print(f"    Optimal threshold : {best_t:.3f}")
    print(f"    Precision         : {p_final:.3f}")
    print(f"    Recall            : {r_final:.3f}")
    print(f"    Accepted / Total  : {n_accept} / {len(pred)}")
    print(f"    Expected trades   : {n_accept}  (from {len(pred)} signal bars in VALID)")
    return best_t


# ══════════════════════════════════════════════════════════════════════
#  SECTION 7 — ML-GATED BACKTEST   (v4 engine + ML filter)
# ══════════════════════════════════════════════════════════════════════

def ml_backtest(df_sig      : pd.DataFrame,
                raw          : pd.DataFrame,
                model        : GradientBoostingClassifier,
                threshold    : float,
                p            : dict,
                split_name   : str) -> pd.DataFrame:
    """
    Runs the full v4 strategy on `df_sig` with one addition:
    a trade is only entered if GBM P(win) >= threshold.

    Everything else — exits, hedges, daily limits, cooldowns — is UNCHANGED.
    """
    # Pre-score every signal bar
    sig_mask = df_sig['signal'].fillna(False)
    sig_idx  = df_sig.index[sig_mask].tolist()
    if len(sig_idx) == 0:
        return pd.DataFrame()

    X_sig = _prep_X(df_sig.loc[sig_mask])
    proba_map = dict(zip(sig_idx, model.predict_proba(X_sig)[:, 1]))
    gate_map  = {i: (proba_map[i] >= threshold) for i in sig_idx}

    # Option lookup (same as v4)
    raw_atm = raw[raw['_off'] == 0][['datetime', 'option_type', 'close']].copy()
    c_lkp = raw_atm[raw_atm['option_type']=='CALL'].set_index('datetime')['close'].to_dict()
    p_lkp = raw_atm[raw_atm['option_type']=='PUT'].set_index('datetime')['close'].to_dict()

    def _opt(dt, flag, fb):
        lkp = c_lkp if flag == 'call' else p_lkp
        v   = lkp.get(dt, 0.0)
        return float(v) if v > 0 else float(fb) / 2.0

    @dataclass
    class _H:
        flag: str; anchor: float; entry_price: float; sl_spot: float
        status: str = 'open'; exit_price: float = 0.0; exit_reason: str = ''

    trades, open_t, entry_i = [], None, None
    daily_count: Dict[date, int] = {}
    last_loss_dt: Optional[pd.Timestamp] = None
    ls, lots, r, cost = p['LOT_SIZE'], p['LOTS'], p['RISK_FREE'], p['STRADDLE_COST']

    df_r = df_sig.reset_index(drop=True)
    for i in range(len(df_r)):
        row   = df_r.iloc[i]
        dt    = row['datetime']
        spot  = row['spot']
        iv    = row.get('iv_dec', 0.0)
        hm    = row['hm']
        strad = row['straddle']
        if pd.isna(spot) or spot <= 0 or pd.isna(iv) or iv <= 0: continue
        if dt.weekday() > 4: continue

        # ── manage open trade ────────────────────────────────────
        if open_t is not None:
            elapsed = (dt - open_t['entry_dt']).total_seconds() / 60.0
            ret     = strad / open_t['entry_straddle'] - 1.0
            iv_r    = iv / open_t['entry_iv'] if open_t['entry_iv'] > 0 else 1.0

            # hedge exits (identical to v4)
            for h in open_t['hedges']:
                if h.status != 'open': continue
                cur = _opt(dt, h.flag, strad)
                rev = abs(spot - h.anchor) <= p['HEDGE_REVERT_PTS']
                sl  = (h.flag == 'put'  and spot >= h.sl_spot) or \
                      (h.flag == 'call' and spot <= h.sl_spot)
                if rev or sl:
                    h.status = 'closed'; h.exit_price = cur
                    h.exit_reason = 'revert' if rev else 'sl'
                    open_t['hedge_pnl'] += (cur - h.entry_price) * lots * ls - p['HEDGE_COST']

            # straddle exit logic (identical to v4)
            ex = None
            if   ret >= p['STRADDLE_TP']:                                    ex = 'gamma_win'
            elif ret <= -p['STRADDLE_SL']:                                   ex = 'max_drawdown'
            elif iv_r >= 1 + p['IV_EXPAND_EXIT']:                           ex = 'vega_win'
            elif elapsed >= p['THETA_MAX_MIN'] and strad < open_t['entry_straddle']: ex = 'theta_bleed'
            elif hm >= p['SESSION_END']:                                     ex = 'eod'

            if ex:
                for h in open_t['hedges']:
                    if h.status != 'open': continue
                    cur = _opt(dt, h.flag, strad)
                    h.status = 'closed'; h.exit_price = cur
                    open_t['hedge_pnl'] += (cur - h.entry_price) * lots * ls - p['HEDGE_COST']

                sp    = (strad - open_t['entry_straddle']) * lots * ls
                total = sp + open_t['hedge_pnl'] - cost
                n_hedges = len(open_t['hedges'])

                trades.append({
                    'split'          : split_name,
                    'entry_dt'       : open_t['entry_dt'],
                    'exit_dt'        : dt,
                    'entry_spot'     : open_t['entry_spot'],
                    'exit_spot'      : spot,
                    'strike'         : open_t['strike'],
                    'dte_entry'      : open_t['dte'],
                    'entry_iv_pct'   : open_t['entry_iv'] * 100,
                    'entry_rv_pct'   : open_t['entry_rv'] * 100,
                    'entry_ivr'      : open_t['entry_ivr'],
                    'ml_prob'        : open_t['ml_prob'],
                    'entry_straddle' : open_t['entry_straddle'],
                    'exit_straddle'  : strad,
                    'straddle_pnl'   : sp,
                    'hedge_pnl'      : open_t['hedge_pnl'],
                    'costs'          : cost,
                    'realized_pnl'   : total,
                    'exit_reason'    : ex,
                    'duration_mins'  : elapsed,
                    'hedge_count'    : n_hedges,
                    'hedge_revert'   : sum(1 for h in open_t['hedges'] if h.exit_reason=='revert'),
                    'hedge_sl'       : sum(1 for h in open_t['hedges'] if h.exit_reason=='sl'),
                    'er_entry'       : open_t['er'],
                })
                if total < 0: last_loss_dt = dt
                open_t = None; entry_i = None
                continue

            # maybe open hedge (identical to v4)
            move = spot - open_t['last_hedge_spot']
            if abs(move) >= p['HEDGE_STEP_PTS']:
                flag = 'put' if move > 0 else 'call'
                sl_s = spot + p['HEDGE_SL_PTS'] if move > 0 else spot - p['HEDGE_SL_PTS']
                pr   = _opt(dt, flag, strad)
                if pr > 0:
                    open_t['hedges'].append(_H(flag=flag, anchor=open_t['last_hedge_spot'],
                                              entry_price=pr, sl_spot=sl_s))
                    open_t['last_hedge_spot'] = spot
            continue

        # ── look for new entry ────────────────────────────────────
        orig_i = df_sig.index[i] if i < len(df_sig) else i
        if not row.get('signal', False): continue
        if not gate_map.get(orig_i, False): continue   # ← ML GATE

        d = dt.date()
        if daily_count.get(d, 0) >= p['MAX_TRADES_PER_DAY']: continue
        if last_loss_dt and dt < last_loss_dt + pd.Timedelta(minutes=p['COOLDOWN_LOSS_MIN']):
            continue
        if strad <= 0: continue

        K = round(spot / 50) * 50
        open_t = {
            'entry_dt'       : dt,
            'entry_spot'     : spot,
            'entry_iv'       : iv,
            'entry_rv'       : float(row.get('rv', iv)),
            'entry_ivr'      : float(row.get('ivr', 50.0)),
            'entry_straddle' : strad,
            'strike'         : K,
            'dte'            : int(row.get('dte', 4)),
            'er'             : float(row.get('er', 0.5)),
            'hedges'         : [],
            'hedge_pnl'      : 0.0,
            'last_hedge_spot': spot,
            'ml_prob'        : proba_map.get(orig_i, 0.0),
        }
        daily_count[d] = daily_count.get(d, 0) + 1

    # close trailing trade
    if open_t is not None:
        last = df_r.iloc[-1]
        for h in open_t['hedges']:
            if h.status != 'open': continue
            cur = _opt(last['datetime'], h.flag, last['straddle'])
            h.status = 'closed'; h.exit_price = cur
            open_t['hedge_pnl'] += (cur - h.entry_price) * lots * ls - p['HEDGE_COST']
        sp    = (last['straddle'] - open_t['entry_straddle']) * lots * ls
        total = sp + open_t['hedge_pnl'] - cost
        trades.append({
            'split': split_name,
            'entry_dt': open_t['entry_dt'], 'exit_dt': last['datetime'],
            'entry_spot': open_t['entry_spot'], 'exit_spot': last['spot'],
            'strike': open_t['strike'], 'dte_entry': open_t['dte'],
            'entry_iv_pct': open_t['entry_iv']*100, 'entry_rv_pct': open_t['entry_rv']*100,
            'entry_ivr': open_t['entry_ivr'], 'ml_prob': open_t['ml_prob'],
            'entry_straddle': open_t['entry_straddle'], 'exit_straddle': last['straddle'],
            'straddle_pnl': sp, 'hedge_pnl': open_t['hedge_pnl'],
            'costs': cost, 'realized_pnl': total, 'exit_reason': 'end_of_data',
            'duration_mins': (last['datetime']-open_t['entry_dt']).total_seconds()/60,
            'hedge_count': len(open_t['hedges']),
            'hedge_revert': sum(1 for h in open_t['hedges'] if h.exit_reason=='revert'),
            'hedge_sl': sum(1 for h in open_t['hedges'] if h.exit_reason=='sl'),
            'er_entry': open_t['er'],
        })

    out = pd.DataFrame(trades)
    if not out.empty:
        out['cumulative_pnl'] = out['realized_pnl'].cumsum()
    return out


# ══════════════════════════════════════════════════════════════════════
#  SECTION 8 — METRICS + REPORTS
# ══════════════════════════════════════════════════════════════════════

def _sharpe(pnl_series: pd.Series) -> float:
    df2 = pd.DataFrame({'pnl': pnl_series, 'dt': pd.to_datetime(pnl_series.index)})
    try:
        daily = pnl_series.reset_index(drop=True)
        return daily.mean() / daily.std() * np.sqrt(252) if daily.std() > 0 else 0.0
    except: return 0.0

def report_split(df: pd.DataFrame, name: str) -> dict:
    if df.empty:
        print(f"\n  {name}: 0 trades (no signals passed ML gate)")
        return {}
    pnl  = df['realized_pnl']
    wins = pnl[pnl > 0]; loss = pnl[pnl <= 0]
    cum  = pnl.cumsum(); mdd  = (cum - cum.cummax()).min()
    df2  = df.copy()
    df2['d'] = pd.to_datetime(df2['entry_dt']).dt.date
    daily   = df2.groupby('d')['realized_pnl'].sum()
    sharpe  = daily.mean()/daily.std()*np.sqrt(252) if daily.std()>0 else 0
    pf      = -wins.sum()/loss.sum() if loss.sum()<0 and wins.sum()>0 else 0
    icon    = '✅' if pnl.sum() > 0 else '❌'

    print(f"\n  {'─'*55}")
    print(f"  {icon}  {name}")
    print(f"  {'─'*55}")
    print(f"    Trades            : {len(df)}")
    print(f"    Win Rate          : {len(wins)/len(pnl)*100:.1f}%")
    print(f"    Total P&L         : ₹{pnl.sum():,.0f}")
    print(f"    Profit Factor     : {pf:.2f}")
    print(f"    Sharpe (ann.)     : {sharpe:.2f}")
    print(f"    Max Drawdown      : ₹{mdd:,.0f}")
    print(f"    Avg Win           : ₹{wins.mean() if len(wins) else 0:,.0f}")
    print(f"    Avg Loss          : ₹{loss.mean() if len(loss) else 0:,.0f}")
    print(f"    Straddle P&L      : ₹{df['straddle_pnl'].sum():,.0f}")
    print(f"    Hedge P&L         : ₹{df['hedge_pnl'].sum():,.0f}")
    print(f"    Total Costs       : ₹{df['costs'].sum():,.0f}")
    print(f"    Total Hedges      : {df['hedge_count'].sum()}")
    print(f"    Avg Hedges/Trade  : {df['hedge_count'].mean():.1f}")
    print(f"    Avg Duration      : {df['duration_mins'].mean():.0f} min")
    print(f"    Avg ML Probability: {df['ml_prob'].mean():.3f}")
    print(f"    Exit breakdown:")
    for r2, cnt in df['exit_reason'].value_counts().items():
        sub = df[df['exit_reason']==r2]
        print(f"      {r2:15s}: {cnt:3d}  avg ₹{sub['realized_pnl'].mean():,.0f}")
    return {'trades': len(df), 'pnl': pnl.sum(), 'wr': len(wins)/len(pnl),
            'pf': pf, 'sharpe': sharpe, 'mdd': mdd}


def evaluate_ml_model(model, name, X, y):
    """Print classification metrics for a split."""
    proba = model.predict_proba(X)[:, 1]
    try:    auc = roc_auc_score(y, proba)
    except: auc = 0.5
    ap = average_precision_score(y, proba)
    pred = (proba >= 0.5).astype(int)
    print(f"\n  {name}  |  ROC-AUC={auc:.3f}  |  AP={ap:.3f}")
    print(classification_report(y, pred, target_names=['loss', 'win'], digits=3, zero_division=0))


# ══════════════════════════════════════════════════════════════════════
#  SECTION 9 — VISUALISATIONS
# ══════════════════════════════════════════════════════════════════════

D='#0d1117'; PAN='#161b22'; G='#39d353'; R='#f85149'
B='#58a6ff'; O='#e3b341';  V='#bc8cff'; GR='#8b949e'; T='#c9d1d9'

def _sty(ax, title=''):
    ax.set_facecolor(PAN)
    ax.tick_params(colors=GR, labelsize=8)
    for sp in ax.spines.values(): sp.set_edgecolor('#30363d')
    if title: ax.set_title(title, color=T, fontsize=9, fontweight='bold', pad=5)
    ax.grid(alpha=0.15, color=GR, lw=0.4)

SPLIT_C = {'TRAIN': B, 'VALID': O, 'TEST': G}

def plot_all(train_t, valid_t, test_t, model, labelled, threshold, out_dir):
    os.makedirs(out_dir, exist_ok=True)

    tr_pnl = train_t['realized_pnl'].sum() if not train_t.empty else 0
    va_pnl = valid_t['realized_pnl'].sum() if not valid_t.empty else 0
    te_pnl = test_t['realized_pnl'].sum()  if not test_t.empty  else 0
    ok     = va_pnl > 0 and te_pnl > 0

    fig = plt.figure(figsize=(26, 30), facecolor=D)
    gs  = gridspec.GridSpec(5, 3, figure=fig, hspace=0.45, wspace=0.30,
                            top=0.93, bottom=0.04, left=0.06, right=0.97)

    fig.text(0.5, 0.962,
        f'ML-ENHANCED GAMMA SCALPING v5.0  ×  GBM + Optuna-style HPT  ×  12-3-3 Walk-Forward\n'
        f'TRAIN ₹{tr_pnl:,.0f}  │  VALID ₹{va_pnl:,.0f}  │  TEST ₹{te_pnl:,.0f}'
        f'  │  Threshold={threshold:.3f}  │  {"✅ Valid+Test PROFIT" if ok else "❌ check threshold"}',
        ha='center', fontsize=11, fontweight='bold',
        color=G if ok else R, fontfamily='monospace')

    # ── 1. Equity curves ──────────────────────────────────────────
    ax = fig.add_subplot(gs[0, :2])
    for nm, t in [('TRAIN', train_t), ('VALID', valid_t), ('TEST', test_t)]:
        if t.empty: continue
        cum = t['realized_pnl'].cumsum()
        col = SPLIT_C[nm]
        ax.fill_between(range(len(cum)), cum, alpha=0.12, color=col)
        ax.plot(cum.values, color=col, lw=2.2, label=f'{nm} ₹{cum.iloc[-1]:,.0f}')
    ax.axhline(0, color=GR, lw=0.5, ls='--')
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v,_: f'₹{v:,.0f}'))
    ax.legend(fontsize=8, facecolor=PAN, labelcolor=T)
    _sty(ax, '📈 Equity Curve — TRAIN (blue)  ·  VALID (orange)  ·  TEST (green)')

    # ── 2. Drawdown ───────────────────────────────────────────────
    ax2 = fig.add_subplot(gs[0, 2])
    all_t = pd.concat([t for t in [train_t, valid_t, test_t] if not t.empty])
    if not all_t.empty:
        cum_a = all_t['realized_pnl'].cumsum()
        dd_a  = cum_a - cum_a.cummax()
        ax2.fill_between(range(len(dd_a)), dd_a, color=R, alpha=0.55)
        ax2.plot(dd_a.values, color=R, lw=0.8)
        ax2.yaxis.set_major_formatter(FuncFormatter(lambda v,_: f'₹{v:,.0f}'))
    _sty(ax2, '📉 Combined Drawdown')

    # ── 3. P&L distribution by split ─────────────────────────────
    ax3 = fig.add_subplot(gs[1, 0])
    for nm, t in [('TRAIN', train_t), ('VALID', valid_t), ('TEST', test_t)]:
        if t.empty: continue
        ax3.hist(t['realized_pnl'].values, bins=20,
                 color=SPLIT_C[nm], alpha=0.55, label=f'{nm} n={len(t)}')
    ax3.axvline(0, color=GR, lw=1, ls='--')
    ax3.legend(fontsize=7, facecolor=PAN, labelcolor=T)
    _sty(ax3, '📊 P&L Distribution by Split')

    # ── 4. Monthly bar chart ──────────────────────────────────────
    ax4 = fig.add_subplot(gs[1, 1:])
    if not all_t.empty:
        at2 = all_t.copy()
        at2['month'] = pd.to_datetime(at2['entry_dt']).dt.to_period('M')
        mon = at2.groupby('month').agg(pnl=('realized_pnl','sum'),
                                        split=('split','first'))
        bar_c = [G if v >= 0 else R for v in mon['pnl']]
        ax4.bar(range(len(mon)), mon['pnl'], color=bar_c, alpha=0.85)
        ax4.set_xticks(range(len(mon)))
        ax4.set_xticklabels([str(m) for m in mon.index], rotation=45, fontsize=7)
        ax4.axhline(0, color=GR, lw=0.5)
        # Annotate split boundaries
        for nm, col in SPLIT_C.items():
            rows = mon[mon['split'] == nm]
            if len(rows):
                ax4.axvspan(list(mon.index).index(rows.index[0]) - 0.4,
                            list(mon.index).index(rows.index[-1]) + 0.4,
                            alpha=0.06, color=col, label=nm)
        ax4.legend(fontsize=7, facecolor=PAN, labelcolor=T)
        ax4.yaxis.set_major_formatter(FuncFormatter(lambda v,_: f'₹{v/1e3:.0f}k'))
    _sty(ax4, '📅 Monthly P&L  (shaded regions = TRAIN/VALID/TEST)')

    # ── 5. Feature importance ─────────────────────────────────────
    ax5 = fig.add_subplot(gs[2, 0])
    imp = model.feature_importances_
    idx = np.argsort(imp)
    ax5.barh(range(len(idx)), imp[idx], color=B, alpha=0.85)
    ax5.set_yticks(range(len(idx)))
    ax5.set_yticklabels([FEATURE_COLS[i] for i in idx], fontsize=6)
    _sty(ax5, '🔍 GBM Feature Importance (all 25)')

    # ── 6. ML probability distribution on labelled samples ────────
    ax6 = fig.add_subplot(gs[2, 1])
    if not labelled.empty:
        X_all = _prep_X(labelled)
        p_all = model.predict_proba(X_all)[:, 1]
        lbl   = labelled['label'].values
        ax6.hist(p_all[lbl==1], bins=25, color=G, alpha=0.7, label='Win')
        ax6.hist(p_all[lbl==0], bins=25, color=R, alpha=0.7, label='Loss')
        ax6.axvline(threshold, color=O, lw=2, ls='--', label=f'Threshold={threshold:.2f}')
        ax6.legend(fontsize=7, facecolor=PAN, labelcolor=T)
    _sty(ax6, '🎯 GBM P(win) Distribution — wins vs losses')

    # ── 7. Precision-Recall curve on VALID ───────────────────────
    ax7 = fig.add_subplot(gs[2, 2])
    if not valid_t.empty and not labelled.empty:
        # build validation labels for PR curve
        val_start = train_t['entry_dt'].max() if not train_t.empty else pd.Timestamp('2025-01-01')
        val_lbl = labelled[labelled['datetime'] > val_start]
        if len(val_lbl) > 5:
            Xv = _prep_X(val_lbl)
            yv = val_lbl['label'].values
            pv = model.predict_proba(Xv)[:, 1]
            prec, rec, _ = precision_recall_curve(yv, pv)
            ap = average_precision_score(yv, pv)
            ax7.plot(rec, prec, color=O, lw=2)
            ax7.axhline(threshold, color=GR, lw=0.8, ls=':', label=f'Prec threshold')
            ax7.set_xlabel('Recall', color=GR); ax7.set_ylabel('Precision', color=GR)
            ax7.text(0.6, 0.1, f'AP={ap:.3f}', color=T, fontsize=9,
                     transform=ax7.transAxes)
            ax7.legend(fontsize=7, facecolor=PAN, labelcolor=T)
    _sty(ax7, '📐 Precision-Recall Curve (VALID)')

    # ── 8. Hedge analysis ─────────────────────────────────────────
    ax8 = fig.add_subplot(gs[3, :2])
    if not all_t.empty:
        ax8.scatter(all_t['hedge_count'], all_t['realized_pnl'],
                    c=[SPLIT_C[s] for s in all_t['split']],
                    s=40, alpha=0.75, zorder=3)
        ax8.axhline(0, color=GR, lw=0.5, ls='--')
        ax8.set_xlabel('Hedge Count per Trade', color=GR)
        ax8.set_ylabel('Realized P&L (₹)', color=GR)
        for nm, col in SPLIT_C.items():
            ax8.scatter([], [], color=col, s=30, label=nm)
        ax8.legend(fontsize=7, facecolor=PAN, labelcolor=T)
    _sty(ax8, '🔄 Hedge Count vs P&L by Split')

    # ── 9. Key metrics table ──────────────────────────────────────
    ax9 = fig.add_subplot(gs[3, 2])
    ax9.axis('off'); ax9.set_facecolor(PAN)

    def _cell(ax, x, y, txt, color=T, bold=False, size=8):
        ax.text(x, y, txt, transform=ax.transAxes, color=color,
                fontsize=size, fontweight='bold' if bold else 'normal')

    _cell(ax9, 0.02, 0.96, '📋 SPLIT COMPARISON', color=T, bold=True, size=9)
    headers = ['Metric', 'TRAIN', 'VALID', 'TEST']
    xs = [0.02, 0.28, 0.55, 0.78]
    for xi, h in zip(xs, headers):
        _cell(ax9, xi, 0.88, h, color=T, bold=True)

    def _row(name, fn, y):
        _cell(ax9, xs[0], y, name, color=GR)
        for xi, (_, t) in zip(xs[1:], [('TRAIN',train_t),('VALID',valid_t),('TEST',test_t)]):
            try:
                v   = fn(t)
                col = G if isinstance(v, (int,float)) and v > 0 else R if isinstance(v,(int,float)) and v < 0 else T
                txt = f'₹{v:,.0f}' if abs(v) > 10 and isinstance(v, float) else str(v) if isinstance(v,int) else f'{v}'
            except: txt, col = 'N/A', GR
            _cell(ax9, xi, y, txt, color=col, bold=True)

    y0 = 0.80
    rows = [
        ('Trades',    lambda t: len(t)),
        ('Win Rate',  lambda t: f"{(t['realized_pnl']>0).mean():.0%}"),
        ('Total P&L', lambda t: t['realized_pnl'].sum()),
        ('PF',        lambda t: round(-t[t['realized_pnl']>0]['realized_pnl'].sum() /
                                       t[t['realized_pnl']<0]['realized_pnl'].sum(), 2)
                                if t[t['realized_pnl']<0]['realized_pnl'].sum() != 0 else 0),
        ('Avg Win',   lambda t: t[t['realized_pnl']>0]['realized_pnl'].mean()),
        ('Avg Loss',  lambda t: t[t['realized_pnl']<0]['realized_pnl'].mean()),
        ('Hedges',    lambda t: t['hedge_count'].sum()),
        ('Avg ML P',  lambda t: f"{t['ml_prob'].mean():.3f}"),
    ]
    for (name, fn) in rows:
        _row(name, fn, y0); y0 -= 0.085

    # ── 10. Win/loss heatmap by exit reason × split ───────────────
    ax10 = fig.add_subplot(gs[4, :])
    if not all_t.empty:
        reasons = sorted(all_t['exit_reason'].unique())
        splits  = ['TRAIN', 'VALID', 'TEST']
        data    = []
        for sp in splits:
            sub = all_t[all_t['split'] == sp]
            data.append([(sub[sub['exit_reason']==r]['realized_pnl']>0).mean()
                          if len(sub[sub['exit_reason']==r]) else np.nan
                          for r in reasons])
        dm = np.array(data, dtype=float)
        im = ax10.imshow(dm, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
        ax10.set_xticks(range(len(reasons)))
        ax10.set_xticklabels(reasons, color=T, fontsize=9)
        ax10.set_yticks(range(3))
        ax10.set_yticklabels(splits, color=T, fontsize=9)
        for i in range(3):
            for j in range(len(reasons)):
                v = dm[i, j]
                if not np.isnan(v):
                    cnt = len(all_t[(all_t['split']==splits[i]) &
                                    (all_t['exit_reason']==reasons[j])])
                    ax10.text(j, i, f'{v:.0%}\n(n={cnt})',
                              ha='center', va='center', color='black',
                              fontsize=8, fontweight='bold')
        plt.colorbar(im, ax=ax10, orientation='horizontal', pad=0.15, label='Win Rate')
    _sty(ax10, '🔥 Win Rate  ×  Exit Reason  ×  Split')

    path = os.path.join(out_dir, 'ml_backtest_v5.png')
    plt.savefig(path, dpi=140, bbox_inches='tight', facecolor=D)
    print(f"\n  Chart saved → {path}")
    plt.close()


# ══════════════════════════════════════════════════════════════════════
#  SECTION 10 — MAIN PIPELINE
# ══════════════════════════════════════════════════════════════════════

def run_ml_pipeline(
        data_1min        : str   = DATA_1MIN,
        out_dir          : str   = OUT_DIR,
        n_hpt_iter       : int   = 200,
        target_precision : float = 0.55,
        min_recall       : float = 0.20,
        seed             : int   = 42,
) -> dict:
    """
    End-to-end ML pipeline.

    Parameters
    ----------
    data_1min        : path to folder with 7 × 1-min CSV files
    out_dir          : output folder for charts and CSVs
    n_hpt_iter       : number of RandomizedSearch iterations (≡ Optuna trials)
    target_precision : minimum precision required at threshold (0.55 = 55%)
    min_recall       : minimum recall (trade rate) required at threshold
    seed             : random seed for reproducibility

    Returns
    -------
    dict with keys: model, threshold, train_trades, valid_trades, test_trades,
                    best_params, cv_score
    """
    os.makedirs(out_dir, exist_ok=True)

    print(f"\n{'═'*62}")
    print(f"  NIFTY GAMMA SCALPING — ML PIPELINE v5.0")
    print(f"  GBM (≡ XGBoost)  +  RandomizedSearch (≡ Optuna)")
    print(f"  12-3-3 Chronological Walk-Forward Split")
    print(f"{'═'*62}")

    # ── Step 1: Load ─────────────────────────────────────────────
    raw   = load_and_prepare(data_1min)
    print(f"\n{'─'*60}\n  Building ATM straddle\n{'─'*60}")
    strad = make_straddle(raw)

    # ── Step 2: Indicators + signals ─────────────────────────────
    print(f"\n{'─'*60}\n  Computing indicators & signals (v4 params, unchanged)\n{'─'*60}")
    sig   = add_indicators(strad, P)
    sig   = add_signals(sig, P)

    # ── Step 3: Feature engineering ──────────────────────────────
    print(f"\n{'─'*60}\n  Building 25-feature matrix (zero lookahead)\n{'─'*60}")
    feat  = build_features(sig)
    print(f"  Feature matrix shape: {feat.shape}")
    print(f"  Features: {FEATURE_COLS}")

    # ── Step 4: Label generation ─────────────────────────────────
    print(f"\n{'─'*60}\n  Generating labels (v4 engine, limits relaxed)\n{'─'*60}")
    labelled = generate_labels(feat, raw, P)

    if len(labelled) < 30:
        print("  ⚠  Fewer than 30 labelled samples — relax IVR/IV_RV filters or extend date range.")
        return {}

    # ── Step 5: 12-3-3 Chronological split ───────────────────────
    tr_lbl, va_lbl, te_lbl = chronological_split(labelled)

    if len(tr_lbl) < 15 or len(va_lbl) < 5:
        print("  ⚠  Train or valid split too small for reliable ML.")
        return {}

    # ── Step 6: Feature matrices ─────────────────────────────────
    X_tr, y_tr = _prep_X(tr_lbl), tr_lbl['label'].values
    X_va, y_va = _prep_X(va_lbl), va_lbl['label'].values
    X_te, y_te = _prep_X(te_lbl), te_lbl['label'].values

    print(f"\n  Shapes  →  Train: {X_tr.shape}  Valid: {X_va.shape}  Test: {X_te.shape}")
    print(f"  WinRate →  Train: {y_tr.mean():.1%}  Valid: {y_va.mean():.1%}  Test: {y_te.mean():.1%}")

    # ── Step 7: Train GBM ────────────────────────────────────────
    print(f"\n{'═'*62}")
    print(f"  TRAINING GBM  ({n_hpt_iter} HPT trials on TRAIN only)")
    print(f"{'═'*62}")
    model, best_params, cv_score = tune_and_train(X_tr, y_tr, n_iter=n_hpt_iter, seed=seed)

    # ── Step 8: Evaluate classifier ──────────────────────────────
    print(f"\n{'─'*60}\n  Classifier Evaluation (before threshold tuning)\n{'─'*60}")
    evaluate_ml_model(model, 'TRAIN', X_tr, y_tr)
    evaluate_ml_model(model, 'VALID', X_va, y_va)
    evaluate_ml_model(model, 'TEST ', X_te, y_te)

    # ── Step 9: Tune threshold on VALID ──────────────────────────
    print(f"\n{'─'*60}\n  Threshold Tuning on VALID (target precision ≥ {target_precision:.0%})\n{'─'*60}")
    threshold = tune_threshold(model, X_va, y_va, target_precision, min_recall)

    # ── Step 10: ML-gated backtest for each split ─────────────────
    print(f"\n{'═'*62}\n  ML-GATED BACKTESTS  (v4 strategy + ML filter)\n{'═'*62}")

    # Date boundaries from labelled data
    tr_end = tr_lbl['datetime'].max()
    va_end = va_lbl['datetime'].max()

    def _slice(df, lo, hi):
        return df[(df['datetime'] > lo) & (df['datetime'] <= hi)].copy() if lo else \
               df[df['datetime'] <= hi].copy()

    sig_tr = feat[feat['datetime'] <= tr_end].copy()
    sig_va = feat[(feat['datetime'] > tr_end) & (feat['datetime'] <= va_end)].copy()
    sig_te = feat[feat['datetime'] > va_end].copy()

    print(f"\n  Running TRAIN backtest  ({sig_tr['datetime'].min().date()} – {sig_tr['datetime'].max().date()})...")
    train_t = ml_backtest(sig_tr, raw, model, threshold, P, 'TRAIN')

    print(f"  Running VALID backtest  ({sig_va['datetime'].min().date()} – {sig_va['datetime'].max().date()})...")
    valid_t = ml_backtest(sig_va, raw, model, threshold, P, 'VALID')

    print(f"  Running TEST  backtest  ({sig_te['datetime'].min().date()} – {sig_te['datetime'].max().date()})...")
    test_t  = ml_backtest(sig_te,  raw, model, threshold, P, 'TEST')

    # ── Step 11: Full performance report ─────────────────────────
    print(f"\n{'═'*62}\n  FINAL PERFORMANCE REPORT\n{'═'*62}")
    m_tr = report_split(train_t, 'TRAIN — 12 months (in-sample)')
    m_va = report_split(valid_t, 'VALID —  3 months (hold-out)')
    m_te = report_split(test_t,  'TEST  —  3 months (unseen)')

    # ── Step 12: Save outputs ────────────────────────────────────
    all_t = pd.concat([t for t in [train_t, valid_t, test_t] if not t.empty],
                      ignore_index=True)
    if not all_t.empty:
        all_t['cumulative_pnl'] = all_t['realized_pnl'].cumsum()

    csv_path = os.path.join(out_dir, 'ml_trades_v5.csv')
    all_t.to_csv(csv_path, index=False)
    print(f"\n  All trades  → {csv_path}  ({len(all_t)} trades total)")

    params_path = os.path.join(out_dir, 'ml_best_params.json')
    with open(params_path, 'w') as f:
        json.dump({
            'cv_score'   : float(cv_score),
            'threshold'  : float(threshold),
            'best_params': {k: (float(v) if isinstance(v, (np.floating, float)) else
                               int(v)   if isinstance(v, (np.integer, int))  else v)
                            for k, v in best_params.items()}
        }, f, indent=2)
    print(f"  Best params → {params_path}")

    model_path = os.path.join(out_dir, 'gbm_model.pkl')
    with open(model_path, 'wb') as f:
        pickle.dump({'model': model, 'threshold': threshold,
                     'feature_cols': FEATURE_COLS}, f)
    print(f"  Model       → {model_path}")

    plot_all(train_t, valid_t, test_t, model, labelled, threshold, out_dir)

    print(f"\n{'═'*62}")
    print(f"  SUMMARY")
    print(f"{'═'*62}")
    for nm, m in [('TRAIN', m_tr), ('VALID', m_va), ('TEST', m_te)]:
        if m:
            icon = '✅' if m.get('pnl', -1) > 0 else '❌'
            print(f"  {icon}  {nm:6s}  ₹{m.get('pnl',0):8,.0f}  |  WR={m.get('wr',0):.0%}"
                  f"  |  Sharpe={m.get('sharpe',0):.2f}  |  Trades={m.get('trades',0)}")

    return {
        'model'       : model,
        'threshold'   : threshold,
        'best_params' : best_params,
        'cv_score'    : cv_score,
        'train_trades': train_t,
        'valid_trades': valid_t,
        'test_trades' : test_t,
        'feature_cols': FEATURE_COLS,
        'labelled'    : labelled,
    }


# ══════════════════════════════════════════════════════════════════════
#  ENTRY POINT
# ══════════════════════════════════════════════════════════════════════

if __name__ == '__main__':
    results = run_ml_pipeline(
        data_1min        = DATA_1MIN,
        out_dir          = OUT_DIR,
        n_hpt_iter       = 400,      # increase to 500 for better tuning (slower)
        target_precision = 0.55,     # 55% precision on VALID = ML only passes high-confidence signals
        min_recall       = 0.18,     # at least 18% of signal bars accepted → enough trades
        seed             = 42,
    )


══════════════════════════════════════════════════════════════
  NIFTY GAMMA SCALPING — ML PIPELINE v5.0
  GBM (≡ XGBoost)  +  RandomizedSearch (≡ Optuna)
  12-3-3 Chronological Walk-Forward Split
══════════════════════════════════════════════════════════════

────────────────────────────────────────────────────────────
  Loading 7 files from: /content/drive/MyDrive/1MIN
  ✓ NIFTY_part_1.csv                     1,000,000 rows
  ✓ NIFTY_part_2.csv                     1,000,000 rows
  ✓ NIFTY_part_3.csv                     1,000,000 rows
  ✓ NIFTY_part_4.csv                     1,000,000 rows
  ✓ NIFTY_part_5.csv                     1,000,000 rows
  ✓ NIFTY_part_6.csv                     1,000,000 rows
  ✓ NIFTY_part_7.csv                        36,829 rows
  Total: 1,438,364 | 2024-08-26 → 2026-02-16

────────────────────────────────────────────────────────────
  Building ATM straddle
────────────────────────────────────────────────────────────
  ATM straddle rows: 155,842 | avg ₹207.9